# 03 - Gold Layer

Create business-ready tables and aggregations.
- Fact tables (pre-joined core business events)
- Aggregated summary tables (daily/monthly metrics)
- Write to `workspace.gold` schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_orders AS
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    c.customer_city,
    c.customer_state,
    i.product_id,
    p.product_category_name_english AS product_category,
    i.price,
    i.freight_value
FROM workspace.silver.orders o
JOIN workspace.silver.customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN workspace.silver.order_items i 
    ON o.order_id = i.order_id
LEFT JOIN workspace.silver.products p 
    ON i.product_id = p.product_id
WHERE o.order_status = 'delivered';

In [0]:
%sql
SELECT * FROM workspace.gold.fact_orders LIMIT 10;

##Create a Daily Sales Summary (Aggregation)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.daily_sales_summary AS
SELECT 
    CAST(order_purchase_timestamp AS DATE) AS sales_date,
    product_category,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS total_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight
FROM workspace.gold.fact_orders
WHERE product_category IS NOT NULL
GROUP BY 
    CAST(order_purchase_timestamp AS DATE),
    product_category
ORDER BY 
    sales_date DESC;

In [0]:
%sql
SELECT * FROM workspace.gold.daily_sales_summary LIMIT 10;